# Workers AI Whisper 對照（MEET 封存測試，免費額度）

把和上次「Whisper 停頓切段」**完全相同的音訊段落**送到 Cloudflare Workers AI 的 `@cf/openai/whisper-large-v3-turbo`，
量錯字率、專名正確率和實際往返延遲，並和上次 Colab GPU 的 Whisper、Chrome 結果放在同一張表。

**不需要 GPU**（執行階段用預設的 CPU 即可）。

### 開始前（只要做一次）

1. **確認是免費方案**：Cloudflare 儀表板 →「Workers 和 Pages」→「方案」顯示 Free。免費方案額度用完只會回錯誤，不會計費。
2. **帳戶 ID**：Cloudflare 儀表板首頁 → 右側「帳戶 ID」（或網址 `dash.cloudflare.com/<這一串>/…`）→ 複製。
3. **API 權杖**：右上角頭像 →「我的設定檔」→「API 權杖」→「建立權杖」→ 範本 **Workers AI** →「使用範本」→「繼續以顯示摘要」→「建立權杖」→ 複製（只會顯示一次）。
4. **存到 Colab 密鑰**：左側欄 🔑「密鑰」→ 新增 `CF_ACCOUNT_ID`、`CF_API_TOKEN` 兩個，貼上值，並打開「筆記本存取權」。
   **不要**把權杖貼在儲存格或對話裡。

### 額度

免費每天 10,000 neurons ≈ 214 分鐘音訊，台灣時間**早上 8 點**重置。本筆記本每次最多送 190 分鐘（第 1 格可改）。
額度用完或達上限會自動暫停，已完成的段落都會存好；隔天把第 1 格 `RESUME` 設成暫停時那個資料夾名稱，再從頭執行一次即可接著跑。

In [ ]:
# 第 1 格：設定
MEET_FOLDER = 'MEET'          # 在「我的雲端硬碟」中的資料夾名稱
ONLY = ''                     # 只評估部分檔案，例如 '01'；空白＝全部
# cfvad10＝停頓切段、每段最長 10 秒（即時字幕）；cfvad30＝每段最長 30 秒（較準、延遲較長）
CF_CONDITIONS = 'cfvad10,cfvad30'
WHISPER_RUN = ''              # 要一起比較的 Colab Whisper 結果資料夾（eval-runs 底下的名稱）；空白＝自動找最新的
RESUME = ''                   # 上次因額度暫停的資料夾名稱；第一次執行留空
CF_MAX_MINUTES = 190          # 這次最多送幾分鐘音訊（免費每天約 214 分鐘）

In [ ]:
# 第 2 格：連接 Google Drive、找到 MEET 與上次的 Whisper 結果
import os, subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive')
MEET = ROOT/MEET_FOLDER
if not MEET.is_dir():
    found = [Path(d)/n for d, dirs, _ in os.walk(ROOT) if len(Path(d).relative_to(ROOT).parts) < 3 for n in dirs if n == MEET_FOLDER]
    if not found:
        raise RuntimeError(f'找不到 {MEET_FOLDER} 資料夾，請確認名稱')
    MEET = found[0]
RUNS = MEET/'eval-runs'
if WHISPER_RUN:
    whisper_run = RUNS/WHISPER_RUN
else:
    candidates = sorted(p for p in RUNS.glob('*') if any(p.glob('*.whisper_live_vad_10s.json'))) if RUNS.is_dir() else []
    whisper_run = candidates[-1] if candidates else None
print('MEET 位置：', MEET)
print('一起比較的 Whisper 結果：', whisper_run or '（沒有，只評估 Workers AI）')
if RESUME:
    if not (RUNS/RESUME).is_dir():
        raise RuntimeError(f'找不到 eval-runs/{RESUME}')
    print('從暫停處繼續：', RUNS/RESUME)

In [ ]:
# 第 3 格：取得固定版本的評估程式與套件（約 1 分鐘）
import sys
COMMIT = '53364807f62b30817eb02af828c43f47e36144d4'
REPO = Path('/content/Meeting-Live-Captions')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/timo893131/Meeting-Live-Captions.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', '-q', 'origin', COMMIT], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '-q', '--detach', COMMIT], cwd=REPO, check=True)
# faster-whisper 只用它的語音偵測（切段），不下載模型
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'faster-whisper==1.2.1', 'av==16.0.1', 'rapidfuzz==3.14.1',
                'opencc-python-reimplemented==0.1.7', 'nodejs-wheel-binaries==22.20.0'], check=True)
import nodejs_wheel
os.environ['NODE_BINARY'] = str(Path(nodejs_wheel.__file__).parent/'bin'/'node')
print('程式版本', COMMIT[:12])

In [ ]:
# 第 4 格：讀取 Colab 密鑰並確認權杖有效（不會印出權杖）
import json, urllib.request, urllib.error
from google.colab import userdata
env = dict(os.environ, CF_ACCOUNT_ID=userdata.get('CF_ACCOUNT_ID').strip(), CF_API_TOKEN=userdata.get('CF_API_TOKEN').strip())
request = urllib.request.Request(f"https://api.cloudflare.com/client/v4/accounts/{env['CF_ACCOUNT_ID']}/tokens/verify",
                                 headers={'Authorization': 'Bearer ' + env['CF_API_TOKEN']})
try:
    with urllib.request.urlopen(request, timeout=30) as response:
        status = json.loads(response.read())['result']['status']
except urllib.error.HTTPError:
    request = urllib.request.Request('https://api.cloudflare.com/client/v4/user/tokens/verify',
                                     headers={'Authorization': 'Bearer ' + env['CF_API_TOKEN']})
    with urllib.request.urlopen(request, timeout=30) as response:
        status = json.loads(response.read())['result']['status']
print('API 權杖狀態：', status)
if status != 'active':
    raise RuntimeError('權杖不是 active，請重新建立')

In [ ]:
# 第 5 格：執行（3 份錄音、兩種條件約 20–40 分鐘，視網路而定）
from datetime import datetime, timezone
RUN = RUNS/datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
conditions = ('full,vad10,' if whisper_run else '') + CF_CONDITIONS
command = [sys.executable, '-u', str(REPO/'scripts/evaluate_meet.py'), '--meet', str(MEET), '--output', str(RUN),
           '--conditions', conditions, '--cf-max-minutes', str(CF_MAX_MINUTES)]
for folder in ([RUNS/RESUME] if RESUME else []) + ([whisper_run] if whisper_run else []):
    command += ['--reuse', str(folder)]
if ONLY:
    command += ['--only', ONLY]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO, env=env)
for line in process.stdout:
    print(line, end='', flush=True)
code = process.wait()
if code == 3:
    print(f'\n⏸ 已暫停。明天早上 8 點後把第 1 格 RESUME 設成 \'{RUN.name}\'，再從第 1 格依序執行。')
elif code:
    raise RuntimeError('評估失敗，請看上面的錯誤訊息')
else:
    print('\n結果已存到：', RUN, '\n請把上面的表格貼回對話。')

## 怎麼看結果

- **workers_ai_live_vad_10s** 對照 **whisper_live_vad_10s**：同一模型、同樣的段落；差異來自 Cloudflare 的執行環境（量化、解碼設定）。
- **workers_ai_live_vad_30s**：每段較長，通常較準但字幕較慢。
- **字幕延遲**＝段長＋Colab 到 Workers AI 的往返時間。Colab 主機多在美國，台灣現場的實際往返通常較短或相近。
- 每行最後的「約 N neurons」是依官方單價估算；實際用量請看 Cloudflare 儀表板 → AI → Workers AI。
- 這是**封存測試**：只用來驗收，不根據它的錯誤調整詞彙或規則。